# 04 — MobileNetV2 transfer learning
This notebook calls the same configurable training/evaluation script as the CLI. It requires local labeled data and TensorFlow; the first training run may download ImageNet MobileNetV2 weights. The base CNN is frozen and only a small classifier head is trained. Scores are not calibrated or deepfake-specific guarantees.

In [ ]:
from pathlib import Path
import json
DATA_DIR = Path('data/images')
MODEL_DIR = Path('models/vision_model')
if not DATA_DIR.exists():
    print('Dataset missing. Follow data/README.md, then rerun this notebook.')
else:
    print('Dataset found:', DATA_DIR.resolve())

Training uses stratified train/validation/test splits, RGB resize, horizontal flip/rotation/zoom augmentation, and `mobilenet_v2.preprocess_input`. Early stopping and best validation-loss checkpointing are enabled. For a fast development experiment, lower epochs and batch size from the command line; the split and evaluation remain real.

In [ ]:
# Execute after confirming the local dataset is present.
if DATA_DIR.exists():
    from src.train_vision import main
    import sys
    _old_argv = sys.argv
    sys.argv = ['train_vision.py', '--data-dir', str(DATA_DIR), '--output-dir', str(MODEL_DIR),
                '--epochs', '4', '--batch-size', '8', '--image-size', '224']
    try:
        main()
    finally:
        sys.argv = _old_argv

In [ ]:
metrics_path = MODEL_DIR / 'metrics.json'
history_path = MODEL_DIR / 'history.json'
if metrics_path.exists() and history_path.exists():
    metrics = json.loads(metrics_path.read_text())
    history = json.loads(history_path.read_text())
    print('Held-out test metrics:', json.dumps(metrics, indent=2))
    import matplotlib.pyplot as plt
    import seaborn as sns
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for key in ('loss', 'val_loss'):
        if key in history: axes[0].plot(history[key], label=key)
    for key in ('accuracy', 'val_accuracy'):
        if key in history: axes[1].plot(history[key], label=key)
    axes[0].set_title('Training/validation loss'); axes[1].set_title('Training/validation accuracy')
    for ax in axes[:2]: ax.set_xlabel('Epoch'); ax.legend()
    sns.heatmap(metrics['confusion_matrix_labels_real_synthetic'], annot=True, fmt='d', cmap='Blues',
                xticklabels=['real', 'synthetic'], yticklabels=['real', 'synthetic'], ax=axes[2])
    axes[2].set_title('Held-out test confusion matrix'); axes[2].set_xlabel('Predicted'); axes[2].set_ylabel('Actual')
    plt.tight_layout(); plt.show()
else:
    print('Run the training cell successfully to generate actual evaluation values and plots.')

## Prediction example
After training, `src.image_detector.predict_image(path)` loads `models/vision_model/model.keras` and returns a class plus uncalibrated model score. It will fail clearly if the model file is absent. Inspect per-class precision/recall and confusion matrix before interpreting an output.

In [ ]:
from src.image_detector import predict_image, ModelUnavailableError
from src.preprocessing import iter_dataset_images
try:
    example_path, _ = next(iter(iter_dataset_images(DATA_DIR)))
    prediction = predict_image(example_path, model_type='vision')
    print(prediction)
except (StopIteration, FileNotFoundError, ModelUnavailableError) as exc:
    print(f'Prediction unavailable until data and a trained model are present: {exc}')